# 03 — Milestone 3 Proof-of-Concept Demo

This compact notebook demonstrates the existing CIND820 pipeline end to end using the reusable functions in `src/co2_pipeline.py`.

**Purpose:** evidence of execution for Milestone 3. It does not introduce new modelling choices or replace the detailed analysis in Notebooks 01 and 02.

Pipeline: **data ingestion → modelling-period evidence → one-year-ahead dataset → chronological split → initial model evaluation**.

In [1]:
from pathlib import Path
import sys

# Work whether Jupyter starts in the repository root or in notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.co2_pipeline import run_poc

DATA_PATH = ROOT / "data" / "raw" / "GCB2022v27_MtCO2_flat.csv"
print("Dataset location: data/raw/GCB2022v27_MtCO2_flat.csv")
print("Dataset exists:", DATA_PATH.exists())

Dataset location: data/raw/GCB2022v27_MtCO2_flat.csv
Dataset exists: True


## 1. Run the functional pipeline

One function executes the same preparation and initial-validation logic documented in the detailed notebooks. Keeping the stages in reusable functions makes the proof of concept easier to reproduce and explain.

In [2]:
poc = run_poc(DATA_PATH)
print("POC pipeline completed successfully.")
print("Raw dataset shape:", poc["raw_data"].shape)
print("Complete-case modelling rows:", poc["diagnostics"]["complete_case_rows"])
print("Entities represented:", poc["diagnostics"]["complete_case_entities"])

POC pipeline completed successfully.
Raw dataset shape: (63104, 11)
Complete-case modelling rows: 12632
Entities represented: 223


## 2. Evidence for the 1960 modelling start

The modelling start is chosen from data profiling rather than arbitrarily. The goal is to balance data quality with enough historical coverage for training and chronological validation.

In [3]:
poc["period_assessment"]

   Start Year   Rows  Core Complete Rows  Core Complete %  Zero Total %
0        1900  28304               15320             54.1          32.8
1        1950  16704               14747             88.3          10.2
2        1960  14384               12956             90.1           8.3
3        1980   9744                8904             91.4           6.4
4        1990   7424                6848             92.2           5.0
5        2000   5104                4743             92.9           4.8

## 3. Chronological validation design

The project is a one-year-ahead forecasting task, so observations are not randomly shuffled. Earlier years are used for training, later years for validation, and the most recent years remain reserved for testing.

In [4]:
poc["split_summary"]

        Split  Predictor Start Year  Predictor End Year   Rows  Entities
0    Training                  1960                2014  11329       223
1  Validation                  2015                2017    652       218
2        Test                  2018                2020    651       217

## 4. Initial model evaluation

Milestone 3 requires a working baseline and interpretable initial results. The persistence baseline predicts next year's Total emissions using the current year's Total emissions. The three learned models are evaluated on the same validation period.

In [5]:
comparison = poc["initial_model_comparison"].copy()
comparison[["MAE", "RMSE"]] = comparison[["MAE", "RMSE"]].round(3)
comparison["R2"] = comparison["R2"].round(4)
comparison

                  Model    MAE    RMSE      R2
0  Persistence Baseline  4.806  21.715  0.9993
1     Linear Regression  5.166  27.404  0.9988
2         Random Forest  5.354  26.724  0.9989
3     Gradient Boosting  5.734  26.621  0.9989

## 5. Initial proof-of-concept conclusion

The end-to-end pipeline is functional and generates interpretable validation results. In the first validation window, **persistence is the strongest benchmark on MAE and RMSE**. This shows that year-to-year CO₂ persistence is difficult to beat and justifies broader chronological validation before final model selection.

For the full analysis, tuning, temporal stability checks, final held-out test, multicollinearity analysis, and robustness diagnostics, see `02_model_preparation.ipynb`.